### Setup

Cria o schema e o volume utilizados pelos arquivos brutos e pelas tabelas
das três camadas (bronze, silver, gold).

Ambiente: Databricks Free Edition (sucessora da Community Edition, com
diferenças relevantes de arquitetura):

- Não há criação manual de cluster; o compute é serverless e é atribuído
  automaticamente na primeira execução de célula.
- Unity Catalog é obrigatório, não há `hive_metastore`. Arquivos brutos são
  armazenados em um Volume, não em DBFS de uso geral.

In [ ]:
catalog_name = "workspace"
schema_name = "case_dados"
volume_name = "raw_files"

RAW_PATH = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

Após a execução da célula anterior, os 9 arquivos fonte devem ser enviados
manualmente: Catalog > workspace > case_dados > Volumes > raw_files >
Upload to this volume.

Caso o volume não seja exibido na lista, atualizar o painel do Catalog.

In [ ]:
display(dbutils.fs.ls(RAW_PATH))

In [ ]:
esperados = {
    "erp_pedidos_cabecalho_2025.csv",
    "erp_pedidos_itens_2025.csv",
    "crm_clientes_export.xlsx",
    "cadastro_produtos_api_dump.json",
    "comercial_canais.xlsx",
    "vendedores.csv",
    "legado_regioes_pipe.txt",
    "logistica_entregas.json",
    "atendimento_ocorrencias.ndjson",
}

encontrados = {f.name for f in dbutils.fs.ls(RAW_PATH)}
faltando = esperados - encontrados

if faltando:
    print("Arquivos faltantes:", faltando)
else:
    print("Os 9 arquivos fonte estão presentes no volume.")